In [ ]:
# ==========================================
# 0. INSTALACIÓN DE DEPENDENCIAS (Ejecutar en Colab)
# ==========================================
!pip install biopython

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from Bio import AlignIO
import numpy as np
import os
import re

# ==========================================
# 1. PARÁMETROS DE USUARIO (Modifica esto)
# ==========================================
# Actualiza esto al nombre exacto de tu archivo actual
ALN_FILE = "BHKalign.aln"

# Asegúrate de poner una referencia que SÍ exista en tu archivo actual
REFERENCE_SEQ_NAME = "BHK.Bothrops.atrox.A0A1L8D5Z7"

# Definición de estructuras secundarias (inicio, fin)
ALPHA_HELICES = [(2, 13), (18, 21), (39, 53), (82, 106)]
BETA_SHEETS = [(66, 69), (73, 77)]

# Parámetros de visualización y espaciado
CHARS_PER_LINE = 70
FONT_SIZE = 15
DPI = 300
OUTPUT_IMAGE = "1atroxvsBHK.png"

# --- AJUSTES DE ESPACIADO ---
char_width = 0.25 # Mantiene las letras más separadas horizontalmente

# Nuevos parámetros para controlar el tamaño de la figura automáticamente
horizontal_padding = 4.0
horizontal_scaling = 1.5

# --- OTROS ESPACIADOS ---
line_spacing = 0.28
block_spacing = 1.5
seq_x_start = 4.0

# ==========================================
# 2. DEFINICIÓN DE GRUPOS Y COLORES
# ==========================================
COLORS = {
    'Polar': '#2ca02c',
    'Apolar': '#000000',
    'Acidic': '#d62728',
    'Basic': '#1f77b4'
}

RESIDUE_GROUPS = {
    'G': 'Polar', 'S': 'Polar', 'T': 'Polar', 'Y': 'Polar', 'C': 'Polar', 'Q': 'Polar', 'N': 'Polar',
    'A': 'Apolar', 'V': 'Apolar', 'L': 'Apolar', 'I': 'Apolar', 'P': 'Apolar', 'F': 'Apolar', 'M': 'Apolar', 'W': 'Apolar',
    'D': 'Acidic', 'E': 'Acidic',
    'K': 'Basic', 'R': 'Basic', 'H': 'Basic',
    '-': 'Gap'
}

# ==========================================
# 3. FUNCIONES AUXILIARES
# ==========================================
def clean_clustal_file(filepath):
    """Limpia los números al final del Clustal sin alterar los espacios originales."""
    cleaned_path = "cleaned_" + filepath
    with open(filepath, "r") as f_in, open(cleaned_path, "w") as f_out:
        for line in f_in:
            if line.strip() and not line.startswith(" ") and not line.startswith("*") and not line.startswith("CLUSTAL"):
                line = re.sub(r'\s+\d+\s*$', '\n', line)
            f_out.write(line)
    return cleaned_path

def get_majority_nature(column):
    counts = {'Polar': 0, 'Apolar': 0, 'Acidic': 0, 'Basic': 0}
    for res in column:
        group = RESIDUE_GROUPS.get(res.upper(), 'Gap')
        if group != 'Gap':
            counts[group] += 1
    if sum(counts.values()) == 0:
        return None
    return max(counts, key=counts.get)

def calculate_identity(seq1, seq2):
    matches = 0
    valid_length = 0
    for a, b in zip(seq1, seq2):
        if a == '-' and b == '-':
            continue
        valid_length += 1
        if a == b:
            matches += 1
    if valid_length == 0:
        return 0.0
    return (matches / valid_length) * 100

# ==========================================
# 4. FUNCIÓN PRINCIPAL DE DIBUJO
# ==========================================
def draw_alignment(aln_file, ref_name, helices, sheets, output_file):
    cleaned_aln = clean_clustal_file(aln_file)
    alignment = AlignIO.read(cleaned_aln, "clustal")

    records = list(alignment)
    names = [rec.id.split('/')[0] for rec in records]
    seqs = [str(rec.seq) for rec in records]
    aln_len = alignment.get_alignment_length()
    num_seqs = len(records)

    try:
        ref_idx = names.index(ref_name)
        ref_seq = seqs[ref_idx]
    except ValueError:
        print(f"¡Advertencia! No se encontró '{ref_name}'. Usando la primera secuencia.")
        ref_idx = 0
        ref_seq = seqs[0]

    identities = [calculate_identity(seq, ref_seq) if i != ref_idx else -1 for i, seq in enumerate(seqs)]

    col_natures = []
    col_conservations = []

    for i in range(aln_len):
        col = [s[i] for s in seqs]
        col_natures.append(get_majority_nature(col))

        res_counts = {}
        for res in col:
            if res != '-':
                res_counts[res] = res_counts.get(res, 0) + 1

        if res_counts:
            max_freq = max(res_counts.values()) / num_seqs
        else:
            max_freq = 0.0
        col_conservations.append(max_freq)

    max_name_len = max(len(name) for name in names)
    name_x = 0
    seq_x_start = max_name_len * 0.095 + 0.2
    id_x = seq_x_start + CHARS_PER_LINE * char_width + 0.5

    # --- CÁLCULO DINÁMICO DEL TAMAÑO DE LA FIGURA ---
    num_blocks = (aln_len // CHARS_PER_LINE) + (1 if aln_len % CHARS_PER_LINE != 0 else 0)

    total_content_width_logical = seq_x_start + CHARS_PER_LINE * char_width
    fig_width = total_content_width_logical * horizontal_scaling + horizontal_padding

    fig_height = num_blocks * (num_seqs * line_spacing + 2.0)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=DPI)
    ax.axis('off')

    y_offset = fig_height

    for block in range(num_blocks):
        start_idx = block * CHARS_PER_LINE
        end_idx = min(start_idx + CHARS_PER_LINE, aln_len)
        block_len = end_idx - start_idx

        struct_y = y_offset + 0.9

        ax.plot([seq_x_start, seq_x_start + block_len * char_width], [struct_y, struct_y], color='black', linewidth=1.2, zorder=1)

        for (h_start, h_end) in helices:
            if h_start <= end_idx and h_end >= start_idx + 1:
                s_draw = max(start_idx + 1, h_start)
                e_draw = min(end_idx, h_end)
                x0 = seq_x_start + (s_draw - start_idx - 1) * char_width
                width = (e_draw - s_draw + 1) * char_width

                cap_offset = 0.08
                ax.plot([x0 + cap_offset, x0 + width - cap_offset], [struct_y, struct_y], color='red', linewidth=8, solid_capstyle='round', zorder=5)

        for (b_start, b_end) in sheets:
            if b_start <= end_idx and b_end >= start_idx + 1:
                s_draw = max(start_idx + 1, b_start)
                e_draw = min(end_idx, b_end)
                x0 = seq_x_start + (s_draw - start_idx - 1) * char_width
                width = (e_draw - s_draw + 1) * char_width

                head_len = char_width if e_draw == b_end else 0
                head_wid = 0.3 if e_draw == b_end else 0

                ax.arrow(x0, struct_y, width - head_len, 0,
                         head_width=head_wid, head_length=head_len,
                         fc='lime', ec='lime', width=0.1, length_includes_head=True, zorder=5)

# 4. Numeración superior
        for i in range(start_idx, end_idx):
            if (i + 1) % 10 == 0:
                # Cambiamos -0.25 a -0.55
                ax.text(seq_x_start + (i - start_idx) * char_width + char_width/2, struct_y - 0.55,
                        str(i + 1), ha='center', va='bottom', fontsize=FONT_SIZE-2, color='gray')
                # Cambiamos -0.35 y -0.45 a -0.65 y -0.75
                ax.plot([seq_x_start + (i - start_idx) * char_width + char_width/2,
                         seq_x_start + (i - start_idx) * char_width + char_width/2],
                        [struct_y - 0.65, struct_y - 0.75], color='gray', lw=0.8)

        # 5. Encabezado de identidad
        if block == 0:
            # Cambiamos -0.25 a -0.55
            ax.text(id_x, struct_y - 0.55, 'Identity (%)', fontweight='bold', fontsize=FONT_SIZE)

        for i, name in enumerate(names):
            y_pos = y_offset - (i * line_spacing) - 0.2
            ax.text(name_x, y_pos, name, fontfamily='monospace', fontweight='bold', style='italic', fontsize=FONT_SIZE, va='center')
            id_val = f"{identities[i]:.1f}" if identities[i] != -1 else "-"
            ax.text(id_x + 0.5, y_pos, id_val, fontfamily='monospace', fontweight='bold', fontsize=FONT_SIZE, va='center', ha='center')
            seq_part = seqs[i][start_idx:end_idx]
            for j, res in enumerate(seq_part):
                global_j = start_idx + j
                col_nature = col_natures[global_j]
                cons_score = col_conservations[global_j]
                res_group = RESIDUE_GROUPS.get(res.upper(), 'Gap')

                bg_alpha = 0
                if cons_score == 1.0:
                    bg_alpha = 0.25
                elif cons_score >= 0.8:
                    bg_alpha = 0.15
                elif cons_score >= 0.6:
                    bg_alpha = 0.08

                if bg_alpha > 0 and res != '-':
                    rect_x = seq_x_start + j * char_width - (char_width / 2)
                    rect_y = y_pos - (line_spacing / 2)
                    rect = patches.Rectangle((rect_x, rect_y), char_width, line_spacing,
                                             facecolor='black', alpha=bg_alpha, edgecolor='none', zorder=0)
                    ax.add_patch(rect)

                color = 'black'
                if res_group == col_nature and res != '-':
                    color = COLORS.get(res_group, 'black')
                elif res == '-':
                    color = '#1f77b4'
                ax.text(seq_x_start + j * char_width, y_pos, res, fontfamily='monospace', fontweight='bold', fontsize=FONT_SIZE, color=color, va='center', ha='center', zorder=10)

        y_offset -= (num_seqs * line_spacing + block_spacing)

    plt.tight_layout()
    plt.savefig(output_file, dpi=DPI, bbox_inches='tight')
    print(f"¡Imagen generada con éxito! Guardada como: {output_file}")
    plt.show()

# ==========================================
# 5. EJECUCIÓN
# ==========================================
draw_alignment(ALN_FILE, REFERENCE_SEQ_NAME, ALPHA_HELICES, BETA_SHEETS, OUTPUT_IMAGE)

/tmp/ipykernel_2785/649098035.py:242: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


¡Imagen generada con éxito! Guardada como: 1atroxvsBHK.png
